In [1]:
import pandas as pd

# ==============================================================================
# DATA
# ==============================================================================
df_penyakittidakmenular = pd.read_csv('jumlah_penyakit_tidak_menular.csv')

df_penyakittidakmenular

,id,id_index,kode_provinsi,nama_provinsi,kode_kabupaten_kota,nama_kabupaten_kota,periode_update,kategori,jumlah,satuan,tahun
0,1,11,35,JAWA TIMUR,3579,KOTA BATU,2018,JUMLAH PENYAKIT HIPERTENSI,10234.0,KASUS,2018
1,1,21,35,JAWA TIMUR,3579,KOTA BATU,2018,JUMLAH PENYAKIT DIABETES MELLITIUS,3058.0,KASUS,2018
2,2,32,35,JAWA TIMUR,3578,KOTA SURABAYA,2018,JUMLAH PENYAKIT DIABETES MELLITIUS,175269.0,KASUS,2018
3,2,42,35,JAWA TIMUR,3578,KOTA SURABAYA,2018,JUMLAH PENYAKIT HIPERTENSI,313960.0,KASUS,2018
4,3,53,35,JAWA TIMUR,3577,KOTA MADIUN,2018,JUMLAH PENYAKIT HIPERTENSI,16023.0,KASUS,2018
...,...,...,...,...,...,...,...,...,...,...,...
527,264,528264,35,JAWA TIMUR,3577,KOTA MADIUN,2024,JUMLAH PENYAKIT DIABETES MELLITIUS,11012.0,KASUS,2024
528,265,529265,35,JAWA TIMUR,3578,KOTA SURABAYA,2024,JUMLAH PENYAKIT HIPERTENSI,738280.0,KASUS,2024
529,265,530265,35,JAWA TIMUR,3578,KOTA SURABAYA,2024,JUMLAH PENYAKIT DIABETES MELLITIUS,113051.0,KASUS,2024
530,266,531266,35,JAWA TIMUR,3579,KOTA BATU,2024,JUMLAH PENYAKIT DIABETES MELLITIUS,7552.0,KASUS,2024


DATA CLEANING

In [2]:
#===============================================================================
# DATA CLEANING
#===============================================================================

# -------------------------
# 1. INFO DATA
# -------------------------
print("\n 1. Info Data:")
df_penyakittidakmenular.info()

print("\n Info Data Setelah Konversi ke String (selain jumlah dan tahun)")
kolom_string = [
    'id',
    'id_index',
    'kode_provinsi',
    'kode_kabupaten_kota',
    'periode_update'
]

df_penyakittidakmenular[kolom_string] = df_penyakittidakmenular[kolom_string].astype(str)
df_penyakittidakmenular.info()

# -------------------------
# 2. STANDARDISASI KAB/KOT
# -------------------------
df_penyakittidakmenular['nama_kabupaten_kota'] = (
    df_penyakittidakmenular['nama_kabupaten_kota']
    .str.upper()
    .str.strip()
)

daftar_kabkot = [
    'KABUPATEN PACITAN',
    'KABUPATEN PONOROGO',
    'KABUPATEN TRENGGALEK',
    'KABUPATEN TULUNGAGUNG',
    'KABUPATEN BLITAR',
    'KABUPATEN KEDIRI',
    'KABUPATEN MALANG',
    'KABUPATEN LUMAJANG',
    'KABUPATEN JEMBER',
    'KABUPATEN BANYUWANGI',
    'KABUPATEN BONDOWOSO',
    'KABUPATEN SITUBONDO',
    'KABUPATEN PROBOLINGGO',
    'KABUPATEN PASURUAN',
    'KABUPATEN SIDOARJO',
    'KABUPATEN MOJOKERTO',
    'KABUPATEN JOMBANG',
    'KABUPATEN NGANJUK',
    'KABUPATEN MADIUN',
    'KABUPATEN MAGETAN',
    'KABUPATEN NGAWI',
    'KABUPATEN BOJONEGORO',
    'KABUPATEN TUBAN',
    'KABUPATEN LAMONGAN',
    'KABUPATEN GRESIK',
    'KABUPATEN BANGKALAN',
    'KABUPATEN SAMPANG',
    'KABUPATEN PAMEKASAN',
    'KABUPATEN SUMENEP',
    'KOTA KEDIRI',
    'KOTA BLITAR',
    'KOTA MALANG',
    'KOTA PROBOLINGGO',
    'KOTA PASURUAN',
    'KOTA MOJOKERTO',
    'KOTA MADIUN',
    'KOTA SURABAYA',
    'KOTA BATU'
]

# -------------------------
# 3. CEK KELENGKAPAN KAB/KOT
# -------------------------
tidak_ada = [
    nama for nama in daftar_kabkot
    if nama not in df_penyakittidakmenular['nama_kabupaten_kota'].values
]

print("\n2.Cek Kelengkapan Kabupaten/Kota")

if len(tidak_ada) > 0:
    print("Nama Kabupaten/kota yang tidak ada di dataset:")
    for nama in tidak_ada:
        print(nama)
else:
    print("Semua nama kabupaten/kota tersedia di dataset")

# -------------------------
# 4. CEK DUPLIKAT
# -------------------------
jumlah_duplikat = df_penyakittidakmenular.duplicated().sum()

print("\n3.Cek Data Duplikat")

if jumlah_duplikat > 0:
    print("Jumlah data duplikat:", jumlah_duplikat)
    df_penyakittidakmenular = df_penyakittidakmenular.drop_duplicates()
    print("Berhasil dihapus")
else:
    print("Tidak ada data duplikat")


# -------------------------
# . CEK MISSING VALUE
# -------------------------
missing_value = df_penyakittidakmenular.isnull().sum()

print("\n4.Cek Missing Value")
for kolom in df_penyakittidakmenular.columns:
    indeks_nan = df_penyakittidakmenular[df_penyakittidakmenular[kolom].isna()].index.tolist()

    if len(indeks_nan) > 0:
        print(f"{kolom}: {indeks_nan}")
        

if missing_value.sum() > 0:
    print("Jumlah missing value:", missing_value.sum())
    kolom_numerik = df_penyakittidakmenular.select_dtypes(include='number').columns
    for kolom in kolom_numerik:
        df_penyakittidakmenular[kolom] = df_penyakittidakmenular[kolom].fillna(df_penyakittidakmenular[kolom].median())
    print("Berhasil ditangani")

else:
    print("Tidak ada missing value")



# -------------------------
# 6. CEK OUTLIER (IQR)
# -------------------------
hasil_outlier = []
kolom_numerik = df_penyakittidakmenular.select_dtypes(include='number').columns

print("\n5.Cek Outlier (IQR)")

for kolom in kolom_numerik:

    Q1 = df_penyakittidakmenular[kolom].quantile(0.25)
    Q3 = df_penyakittidakmenular[kolom].quantile(0.75)

    IQR = Q3 - Q1

    batas_bawah = Q1 - 1.5 * IQR
    batas_atas = Q3 + 1.5 * IQR

    jumlah_outlier = df_penyakittidakmenular[
        (df_penyakittidakmenular[kolom] < batas_bawah) |
        (df_penyakittidakmenular[kolom] > batas_atas)
    ]


    if len(jumlah_outlier) > 0:
        keterangan = "Ada outlier"
        print(f"\nVariabel: {kolom}")
        print(f'Jumlah Outlier: {len(jumlah_outlier)}')
        display(jumlah_outlier[['nama_kabupaten_kota', 'tahun', kolom]])
    else:
        print(f"\nVariabel: {kolom}")
        print("Tidak ada outlier")


 1. Info Data:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 532 entries, 0 to 531
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   532 non-null    int64  
 1   id_index             532 non-null    int64  
 2   kode_provinsi        532 non-null    int64  
 3   nama_provinsi        532 non-null    object 
 4   kode_kabupaten_kota  532 non-null    int64  
 5   nama_kabupaten_kota  532 non-null    object 
 6   periode_update       532 non-null    int64  
 7   kategori             532 non-null    object 
 8   jumlah               532 non-null    float64
 9   satuan               532 non-null    object 
 10  tahun                532 non-null    int64  
dtypes: float64(1), int64(6), object(4)
memory usage: 45.8+ KB

 Info Data Setelah Konversi ke String (selain jumlah dan tahun)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 532 entries, 0 to 531
Data columns (total 11 column

,nama_kabupaten_kota,tahun,jumlah
3,KOTA SURABAYA,2018,313960.0
51,KABUPATEN PASURUAN,2018,210560.0
65,KABUPATEN MALANG,2018,201368.0
92,KABUPATEN JEMBER,2019,366860.0
95,KABUPATEN BANYUWANGI,2019,343326.0
...,...,...,...
502,KABUPATEN LAMONGAN,2024,338504.0
509,KABUPATEN SAMPANG,2024,239371.0
510,KABUPATEN PAMEKASAN,2024,230393.0
513,KABUPATEN SUMENEP,2024,258808.0



Variabel: tahun
Tidak ada outlier


TRANSFORMASI DATA

In [3]:
# ==============================================================================
# AGGREGASI JUMLAH PENYAKIT TIDAK MENULAR PER KABUPATEN/KOTA PER TAHUN
# ==============================================================================

# -------------------------
# JUMLAH PENYAKIT TIDAK MENULAR PER KABUPATEN/KOTA PER TAHUN
# -------------------------
df_penyakittidakmenular_tahun_kab = (
    df_penyakittidakmenular.groupby([
        'kode_kabupaten_kota',
        'nama_kabupaten_kota',
        'tahun'
    ])['jumlah']
    .sum()
    .reset_index()
)
df_penyakittidakmenular_tahun_kab = df_penyakittidakmenular_tahun_kab.rename(columns={
    'jumlah': 'JUMLAH_PENYAKIT_TIDAK_MENULAR'
})

df_penyakittidakmenular_tahun_kab = df_penyakittidakmenular_tahun_kab.sort_values(
    ['tahun', 'kode_kabupaten_kota'],
    ascending=[True, True]
).reset_index(drop=True)

df_penyakittidakmenular_tahun_kab['JUMLAH_PENYAKIT_TIDAK_MENULAR'] = (
    df_penyakittidakmenular_tahun_kab['JUMLAH_PENYAKIT_TIDAK_MENULAR']
    .round()
    .astype('Int64')
)

df_penyakittidakmenular_tahun_kab



,kode_kabupaten_kota,nama_kabupaten_kota,tahun,JUMLAH_PENYAKIT_TIDAK_MENULAR
0,3501,KABUPATEN PACITAN,2018,41173
1,3502,KABUPATEN PONOROGO,2018,46778
2,3503,KABUPATEN TRENGGALEK,2018,40464
3,3504,KABUPATEN TULUNGAGUNG,2018,114929
4,3505,KABUPATEN BLITAR,2018,65232
...,...,...,...,...
261,3575,KOTA PASURUAN,2024,18844
262,3576,KOTA MOJOKERTO,2024,16010
263,3577,KOTA MADIUN,2024,23764
264,3578,KOTA SURABAYA,2024,851331


SIMPAN DATA

In [4]:
#===============================================================================
# SIMPAN DATA
#===============================================================================
df_penyakittidakmenular_tahun_kab.to_csv('data_jumlah_penyakittidakmenular.csv', index=False)

In [5]:
print(df_penyakittidakmenular_tahun_kab.columns.tolist())

['kode_kabupaten_kota', 'nama_kabupaten_kota', 'tahun', 'JUMLAH_PENYAKIT_TIDAK_MENULAR']
